# TP KBO/BCE — Notebook complété

⚠️ Exécute d’abord la cellule d’initialisation ci-dessous, puis lance le reste du notebook dans l’ordre. Dans VS Code : **Restart Kernel** puis **Run All**.


In [1]:
# ============================================================
# Initialisation du TP : imports, constantes, chargement KBO
# ============================================================
from __future__ import annotations

import json
import re
import time
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception:
    px = None
    go = None

GOOGLE_NUM = "0878.065.378"
APPLE_NUM  = "0836.157.420"
SNCB_NUM   = "0203.430.576"

ENTREPRISES = {
    "Google Belgium": GOOGLE_NUM,
    "Apple Belgium": APPLE_NUM,
    "SNCB": SNCB_NUM,
}

KBO_DIR_CANDIDATES = [
    Path("/home/jovyan/work/data/KBO"),
    Path("/data/kbo"),
    Path("./data/KBO"),
    Path("./KBO"),
]

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 student-project KBO/BCE TP",
    "Accept-Language": "fr-FR,fr;q=0.9,nl;q=0.8,en;q=0.7",
}


def normalize_num(num: str) -> str:
    """Garde uniquement les chiffres du numéro d'entreprise."""
    return re.sub(r"\D", "", str(num))


def dotted_num(num: str) -> str:
    """Convertit 878065378 ou 0878065378 en 0878.065.378."""
    n = normalize_num(num).zfill(10)
    return f"{n[:4]}.{n[4:7]}.{n[7:]}"


def num_for_url(num: str) -> str:
    """Format attendu par les URL publiques belges : sans points et sans zéro initial."""
    return normalize_num(num).lstrip("0")


def find_kbo_dir() -> Optional[Path]:
    for p in KBO_DIR_CANDIDATES:
        if p.exists():
            return p
    return None


def read_csv_flexible(path: Path) -> pd.DataFrame:
    """Lit un CSV en testant plusieurs séparateurs/encodages courants."""
    for sep in [",", ";", "\t", "|"]:
        for enc in ["utf-8", "utf-8-sig", "latin1"]:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, dtype=str, low_memory=False)
                if df.shape[1] > 1:
                    df.columns = [str(c).strip() for c in df.columns]
                    return df.fillna("")
            except Exception:
                pass
    raise RuntimeError(f"Impossible de lire {path}. Vérifie le séparateur et l'encodage.")


def load_kbo_tables(kbo_dir: Optional[Path] = None) -> dict[str, pd.DataFrame]:
    """Charge les fichiers KBO. Si les fichiers ne sont pas disponibles, crée des DataFrames vides."""
    kbo_dir = kbo_dir or find_kbo_dir()
    files = {
        "enterprise": "enterprise.csv",
        "denomination": "denomination.csv",
        "address": "address.csv",
        "activity": "activity.csv",
        "contact": "contact.csv",
        "establishment": "establishment.csv",
        "code": "code.csv",
    }
    tables = {}
    if kbo_dir is None:
        print("⚠️ Aucun dossier KBO trouvé. Place les CSV dans ./data/KBO ou utilise le chemin JupyterHub demandé.")
        return {k: pd.DataFrame() for k in files}

    print(f"Dossier KBO utilisé : {kbo_dir}")
    for key, filename in files.items():
        path = kbo_dir / filename
        if not path.exists():
            print(f"⚠️ Fichier manquant : {path}")
            tables[key] = pd.DataFrame()
        else:
            tables[key] = read_csv_flexible(path)
            print(f"✅ {filename:<20} {tables[key].shape[0]:>8} lignes | {tables[key].shape[1]:>3} colonnes")
    return tables


def find_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    """Retrouve une colonne malgré les variations de casse ou de nom."""
    if df.empty:
        return None
    normalized = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in df.columns}
    for cand in candidates:
        key = re.sub(r"[^a-z0-9]", "", cand.lower())
        if key in normalized:
            return normalized[key]
    for c in df.columns:
        lc = re.sub(r"[^a-z0-9]", "", c.lower())
        if any(re.sub(r"[^a-z0-9]", "", cand.lower()) in lc for cand in candidates):
            return c
    return None


def filter_by_enterprise(df: pd.DataFrame, num: str) -> pd.DataFrame:
    """Filtre une table KBO sur le numéro d'entreprise."""
    if df.empty:
        return df.copy()

    plain = normalize_num(num).lstrip("0")
    dotted = dotted_num(num)
    possible_cols = [
        "EnterpriseNumber", "enterpriseNumber", "EntityNumber", "entityNumber",
        "Ondernemingsnummer", "Numéro d'entreprise", "NumeroEntreprise",
        "EstablishmentNumber", "establishmentNumber",
    ]
    col = find_column(df, possible_cols)

    if col is not None:
        s = df[col].astype(str)
        # Comparaison vectorisee directe au format pointe (0878.065.378) : pas de regex
        # sur des dizaines de millions de lignes -> evite le MemoryError sur activity.csv.
        mask = s == dotted
        if not mask.any():
            # Repli peu couteux : on retire seulement les points (str.replace litteral, niveau C).
            mask = s.str.replace(".", "", regex=False).str.lstrip("0") == plain
        return df[mask].copy()

    # Fallback : aucune colonne de numero reconnue -> on balaie les colonnes une a une.
    # Meme precaution que ci-dessus : pas de regex \D sur des dizaines de millions de
    # lignes. On compare d'abord au format pointe, repli litteral (niveau C) si besoin,
    # et on s'arrete des qu'une colonne porte le numero.
    mask = pd.Series(False, index=df.index)
    for c in df.columns:
        s = df[c].astype(str)
        hit = s == dotted
        if not hit.any():
            hit = s.str.replace(".", "", regex=False).str.lstrip("0") == plain
        if hit.any():
            mask = hit
            break
    return df[mask].copy()


def translate_value(value: Any, code_table: pd.DataFrame, code_type: str = "") -> Any:
    """Traduit un code à l'aide de code.csv si les colonnes nécessaires existent."""
    if code_table.empty or value in [None, ""]:
        return value
    code_col = find_column(code_table, ["Code", "code"])
    desc_col = find_column(code_table, ["Description", "DescriptionFR", "DescriptionNL", "Label", "Libelle"])
    type_col = find_column(code_table, ["Category", "Type", "CodeType", "CodeList"])
    if not code_col or not desc_col:
        return value
    temp = code_table.copy()
    if code_type and type_col:
        temp = temp[temp[type_col].astype(str).str.contains(code_type, case=False, na=False)]
    hit = temp[temp[code_col].astype(str).str.strip() == str(value).strip()]
    return hit.iloc[0][desc_col] if not hit.empty else value


def get_entity(num: str, tables_arg: Optional[dict[str, pd.DataFrame]] = None) -> dict[str, Any]:
    """Construit l'entité depuis les CSV KBO."""
    t = tables_arg if tables_arg is not None else tables
    return {
        "numero": dotted_num(num),
        "enterprise": filter_by_enterprise(t.get("enterprise", pd.DataFrame()), num),
        "denominations": filter_by_enterprise(t.get("denomination", pd.DataFrame()), num),
        "addresses": filter_by_enterprise(t.get("address", pd.DataFrame()), num),
        "activities": filter_by_enterprise(t.get("activity", pd.DataFrame()), num),
        "contacts": filter_by_enterprise(t.get("contact", pd.DataFrame()), num),
        "establishments": filter_by_enterprise(t.get("establishment", pd.DataFrame()), num),
    }


def afficher_entity(entity: dict[str, Any], n: int = 10) -> None:
    """Affiche proprement les sous-tables liées à une entreprise."""
    display(Markdown(f"### Entité {entity['numero']}"))
    for key, value in entity.items():
        if key == "numero":
            continue
        display(Markdown(f"**{key}**"))
        if isinstance(value, pd.DataFrame):
            if value.empty:
                print("Aucune donnée trouvée.")
            else:
                display(value.head(n))
        else:
            print(value)


def export_entity(entity: dict[str, Any], name: str) -> None:
    folder = OUTPUT_DIR / re.sub(r"\W+", "_", name.lower()).strip("_")
    folder.mkdir(parents=True, exist_ok=True)
    for key, value in entity.items():
        if isinstance(value, pd.DataFrame) and not value.empty:
            value.to_csv(folder / f"{key}.csv", index=False)

# Chargement effectif des tables KBO
tables = load_kbo_tables()


Dossier KBO utilisé : data\KBO


✅ enterprise.csv        1951671 lignes |   7 colonnes


✅ denomination.csv      3346292 lignes |   4 colonnes


✅ address.csv           2881017 lignes |  13 colonnes


✅ activity.csv         34510460 lignes |   5 colonnes


✅ contact.csv            706353 lignes |   4 colonnes


✅ establishment.csv     1687516 lignes |   3 colonnes
✅ code.csv                21468 lignes |   4 colonnes


## 1) Création de l'entité initiale

On commence avec un catalogue de fichiers CSV qui contiennent les données de nos entreprises. On cherche à trouver, stocker et analyser les données des trois entreprises : **Google** (`0878.065.378`), **Apple** (`0836.157.420`) et **SNCB** (`0203.430.576`).

Les fichiers KBO Open Data se trouvent dans `/home/jovyan/work/data/KBO/` :

| Fichier | Contenu |
|---|---|
| `enterprise.csv` | Statut, forme juridique, date de création |
| `denomination.csv` | Dénominations officielles (FR / NL) |
| `address.csv` | Adresses (siège, exploitation…) |
| `activity.csv` | Codes d'activité NACE |
| `contact.csv` | Téléphone, e-mail, site web |
| `establishment.csv` | Unités d'établissement |
| `code.csv` | Table de correspondance des codes → libellés |

### 🔍 Recherche — Google

In [2]:
google = get_entity(GOOGLE_NUM)
afficher_entity(google)

### Entité 0878.065.378

**enterprise**

,EnterpriseNumber,Status,JuridicalSituation,TypeOfEnterprise,JuridicalForm,JuridicalFormCAC,StartDate
1519146,0878.065.378,AC,000,2,014,,21-12-2005


**denominations**

,EntityNumber,Language,TypeOfDenomination,Denomination
1618853,0878.065.378,2,001,GOOGLE BELGIUM


...

,EntityNumber,ActivityGroup,NaceVersion,NaceCode,Classification
5359873,0878.065.378,006,2025,73110,MAIN
5359874,0878.065.378,006,2003,74401,MAIN
5359875,0878.065.378,001,2003,72600,MAIN
5359876,0878.065.378,001,2025,62900,MAIN
5359877,0878.065.378,006,2008,73110,MAIN
5359878,0878.065.378,001,2008,62090,MAIN


**contacts**

Aucune donnée trouvée.


**establishments**

,EstablishmentNumber,StartDate,EnterpriseNumber
342247,2.151.627.472,10-02-2006,0878.065.378


### 🔍 Recherche — Apple

In [3]:
apple = get_entity(APPLE_NUM)
afficher_entity(apple)

### Entité 0836.157.420

**enterprise**

,EnterpriseNumber,Status,JuridicalSituation,TypeOfEnterprise,JuridicalForm,JuridicalFormCAC,StartDate
1343357,0836.157.420,AC,000,2,610,,06-05-2011


**denominations**

,EntityNumber,Language,TypeOfDenomination,Denomination
1435097,0836.157.420,2,001,APPLE RETAIL BELGIUM


...

,EntityNumber,ActivityGroup,NaceVersion,NaceCode,Classification
4838762,0836.157.420,006,2025,47400,MAIN
4838763,0836.157.420,001,2025,47400,MAIN
4838764,0836.157.420,006,2008,47410,MAIN
4838765,0836.157.420,001,2008,47410,MAIN


**contacts**

Aucune donnée trouvée.


**establishments**

,EstablishmentNumber,StartDate,EnterpriseNumber
526652,2.200.008.005,06-05-2011,0836.157.420
716874,2.244.110.935,19-09-2015,0836.157.420


### 🔍 Recherche — SNCB

In [4]:
sncb = get_entity(SNCB_NUM)
afficher_entity(sncb)

### Entité 0203.430.576

**enterprise**

,EnterpriseNumber,Status,JuridicalSituation,TypeOfEnterprise,JuridicalForm,JuridicalFormCAC,StartDate
62,0203.430.576,AC,000,2,114,,01-01-1968


**denominations**

,EntityNumber,Language,TypeOfDenomination,Denomination
113,0203.430.576,1,001,SOCIÉTÉ NATIONALE DES CHEMINS DE FER BELGES
114,0203.430.576,1,002,SNCB
115,0203.430.576,2,001,NATIONALE MAATSCHAPPIJ DER BELGISCHE SPOORWEGEN
116,0203.430.576,2,002,NMBS


...

,EntityNumber,ActivityGroup,NaceVersion,NaceCode,Classification
236,0203.430.576,001,2025,80010,SECO
237,0203.430.576,001,2025,49110,MAIN
238,0203.430.576,001,2003,74601,SECO
239,0203.430.576,001,2008,80100,SECO
240,0203.430.576,001,2008,49100,MAIN


**contacts**

Aucune donnée trouvée.


**establishments**

,EstablishmentNumber,StartDate,EnterpriseNumber
314967,2.143.271.418,01-01-1968,0203.430.576
314968,2.143.272.111,01-01-1968,0203.430.576
314969,2.143.273.594,01-01-1968,0203.430.576
314970,2.143.273.693,01-01-1968,0203.430.576
314971,2.143.273.891,01-01-1968,0203.430.576
314972,2.143.274.584,01-01-1968,0203.430.576
314973,2.143.274.683,01-01-1968,0203.430.576
314974,2.143.274.782,01-01-1968,0203.430.576
314975,2.143.274.980,01-01-1968,0203.430.576
314976,2.143.275.079,01-01-1968,0203.430.576


### 🔄 Traduction des codes CSV

On traduit maintenant les codes issus des fichiers CSV en valeurs lisibles (libellés, catégories, etc.).

In [5]:
# ============================================================
# Traduction des codes CSV avec code.csv
# ============================================================
code_table = tables.get("code", pd.DataFrame())

def enrich_entity_codes(entity: dict[str, Any]) -> dict[str, Any]:
    """Ajoute une version enrichie des sous-tables avec les colonnes *_libelle quand possible."""
    enriched = {"numero": entity["numero"]}
    for name, df in entity.items():
        if name == "numero" or not isinstance(df, pd.DataFrame) or df.empty:
            enriched[name] = df
            continue
        tmp = df.copy()
        # On tente une traduction des colonnes qui ressemblent à des codes.
        for col in tmp.columns:
            if re.search(r"code|type|status|jurid|legal|nace|activity|function", col, re.I):
                new_col = f"{col}_libelle"
                tmp[new_col] = tmp[col].apply(lambda x: translate_value(x, code_table))
        enriched[name] = tmp
    return enriched

entities = {
    "Google Belgium": enrich_entity_codes(google),
    "Apple Belgium": enrich_entity_codes(apple),
    "SNCB": enrich_entity_codes(sncb),
}

for name, entity in entities.items():
    export_entity(entity, name)
    print(f"✅ Export CSV réalisé pour {name} dans le dossier outputs/.")


✅ Export CSV réalisé pour Google Belgium dans le dossier outputs/.
✅ Export CSV réalisé pour Apple Belgium dans le dossier outputs/.
✅ Export CSV réalisé pour SNCB dans le dossier outputs/.


In [6]:
# Résumé après traduction / transformation
resume_rows = []
for name, entity in entities.items():
    row = {"entreprise": name, "numero": entity["numero"]}
    for section, df in entity.items():
        if section != "numero" and isinstance(df, pd.DataFrame):
            row[f"nb_{section}"] = len(df)
    resume_rows.append(row)

resume_entities = pd.DataFrame(resume_rows)
display(resume_entities)


,entreprise,numero,nb_enterprise,nb_denominations,nb_addresses,nb_activities,nb_contacts,nb_establishments
0,Google Belgium,0878.065.378,1,1,1,6,0,1
1,Apple Belgium,0836.157.420,1,1,1,4,0,2
2,SNCB,0203.430.576,1,4,1,5,0,270


---
## 2) Informations supplémentaires externes (KBO)

À partir d'ici, on récupère des données plus riches depuis le **Carrefour des Entreprises Belges (KBO/BCE)**.

URL de référence :  
`https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer=<NUMERO>`

Exemple : https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer=203430576

In [7]:
# ============================================================
# Scraping KBO/BCE Public Search
# ============================================================
GOOGLE_NUM_URL = num_for_url(GOOGLE_NUM)
APPLE_NUM_URL  = num_for_url(APPLE_NUM)
SNCB_NUM_URL   = num_for_url(SNCB_NUM)

BASE_URL = "https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html"


def clean_text(x: str) -> str:
    return re.sub(r"\s+", " ", x or "").strip()


def get_soup(url: str, params: Optional[dict[str, Any]] = None) -> BeautifulSoup:
    r = requests.get(url, params=params, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def kbo_public_url(num: str) -> str:
    return f"{BASE_URL}?ondernemingsnummer={num_for_url(num)}"


def scrape_kbo_page(num: str) -> dict[str, Any]:
    """Récupère la page KBO Public Search et extrait les tableaux/clé-valeurs."""
    url = kbo_public_url(num)
    soup = get_soup(url)

    rows = []
    for tr in soup.select("tr"):
        cells = [clean_text(td.get_text(" ")) for td in tr.find_all(["td", "th"])]
        if len(cells) >= 2:
            rows.append(cells)

    kv = {}
    for cells in rows:
        key = cells[0].rstrip(":")
        value = " | ".join(cells[1:])
        if key and value:
            kv.setdefault(key, value)

    tables_html = pd.read_html(str(soup)) if soup.find("table") else []
    return {
        "numero": dotted_num(num),
        "url": url,
        "champs": kv,
        "tables": tables_html,
        "texte": clean_text(soup.get_text(" ")),
    }


def safe_scrape_kbo(num: str) -> dict[str, Any]:
    try:
        return scrape_kbo_page(num)
    except Exception as e:
        return {"numero": dotted_num(num), "url": kbo_public_url(num), "champs": {}, "tables": [], "texte": "", "erreur": str(e)}

# Scraping des trois entreprises. Si le site bloque l'accès, l'erreur sera conservée dans le dictionnaire.
kbo_pages = {name: safe_scrape_kbo(num) for name, num in ENTREPRISES.items()}

for name, data in kbo_pages.items():
    print(f"{name} -> {data['url']}")
    if data.get("erreur"):
        print("  ⚠️", data["erreur"])
    else:
        print(f"  ✅ {len(data['champs'])} champs et {len(data['tables'])} tableaux extraits")


C:\Users\louis\AppData\Local\Temp\ipykernel_16880\3921175108.py:43: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_html = pd.read_html(str(soup)) if soup.find("table") else []


C:\Users\louis\AppData\Local\Temp\ipykernel_16880\3921175108.py:43: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_html = pd.read_html(str(soup)) if soup.find("table") else []


Google Belgium -> https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer=878065378
  ✅ 18 champs et 2 tableaux extraits
Apple Belgium -> https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer=836157420
  ✅ 17 champs et 2 tableaux extraits
SNCB -> https://kbopub.economie.fgov.be/kbopub/toonondernemingps.html?ondernemingsnummer=203430576
  ✅ 21 champs et 2 tableaux extraits


C:\Users\louis\AppData\Local\Temp\ipykernel_16880\3921175108.py:43: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables_html = pd.read_html(str(soup)) if soup.find("table") else []


---
### 2.1) Informations Générales

| Champ | Valeur |
|---|---|
| Dénomination | |
| Numéro d'entreprise | |
| Adresse | |
| Activité principale | |
| Effectif | |
| Date de création | |

In [8]:
# Extraction — Informations Générales
GENERAL_PATTERNS = {
    "Dénomination": r"dénomination|denomination|naam|nom",
    "Numéro d'entreprise": r"numéro d'entreprise|ondernemingsnummer|enterprise number",
    "Adresse": r"adresse|address|adres|siège|zetel",
    "Activité principale": r"activité|activiteit|nace|onss",
    "Effectif": r"effectif|personnel|employ|werknemers",
    "Date de création": r"date de création|constitution|startdatum|création",
}


def pick_kbo_field(data: dict[str, Any], pattern: str) -> str:
    for k, v in data.get("champs", {}).items():
        if re.search(pattern, k, flags=re.I):
            return v
    text = data.get("texte", "")
    m = re.search(pattern + r"\s*[:\-]?\s*(.{0,180})", text, flags=re.I)
    return clean_text(m.group(1)) if m else ""

infos_generales = []
for name, data in kbo_pages.items():
    row = {"Entreprise": name}
    for champ, pattern in GENERAL_PATTERNS.items():
        row[champ] = pick_kbo_field(data, pattern)
    # fallback utile depuis les CSV locaux
    row["Numéro d'entreprise"] = row["Numéro d'entreprise"] or data.get("numero", "")
    infos_generales.append(row)

infos_generales_df = pd.DataFrame(infos_generales)
display(infos_generales_df)


,Entreprise,Dénomination,Numéro d'entreprise,Adresse,Activité principale,Effectif,Date de création
0,Google Belgium,"GOOGLE BELGIUM Naam in het Nederlands, sinds 1...",0878.065.378,Steenweg op Etterbeek 180 1040 Brussel Extra a...,,,
1,Apple Belgium,"APPLE RETAIL BELGIUM Naam in het Nederlands, s...",0836.157.420,Sint-Lazaruslaan 4-10 1210 Sint-Joost-ten-Node...,,,
2,SNCB,NATIONALE MAATSCHAPPIJ DER BELGISCHE SPOORWEGE...,0203.430.576,Frankrijkstraat 56 1060 Sint-Gillis (bij-Bruss...,,,


---
### 2.2) Informations Juridiques

| Champ | Valeur |
|---|---|
| Forme juridique | |
| Numéro de TVA | |
| Situation juridique | |
| Capital social | |
| Assemblée générale | |
| Date de fin de l'année comptable | |

In [9]:
# Extraction — Informations Juridiques
LEGAL_PATTERNS = {
    "Forme juridique": r"forme juridique|rechtsvorm|legal form",
    "Numéro de TVA": r"tva|btw|vat",
    "Situation juridique": r"situation juridique|rechtstoestand|statut|status",
    "Capital social": r"capital|kapitaal",
    "Assemblée générale": r"assemblée générale|algemene vergadering",
    "Date de fin de l'année comptable": r"fin de l'année comptable|einde boekjaar|financial year end|clôture",
}

infos_juridiques = []
for name, data in kbo_pages.items():
    row = {"Entreprise": name}
    for champ, pattern in LEGAL_PATTERNS.items():
        row[champ] = pick_kbo_field(data, pattern)
    infos_juridiques.append(row)

infos_juridiques_df = pd.DataFrame(infos_juridiques)
display(infos_juridiques_df)


,Entreprise,Forme juridique,Numéro de TVA,Situation juridique,Capital social,Assemblée générale,Date de fin de l'année comptable
0,Google Belgium,Naamloze vennootschap Sinds 19 december 2005,,Actief,"770.000,00 EUR",,
1,Apple Belgium,Besloten Vennootschap Sinds 1 januari 2020,,Actief,,,
2,SNCB,Naamloze vennootschap van publiek recht Sinds ...,,Actief,"249.022.345,57 EUR",,


---
### 2.3) Activités

Liste de chaque domaine d'activité avec son code NACE/ONSS.

| Code | Description |
|---|---|

In [10]:
# Extraction — Activités
# 1) Depuis les CSV KBO si disponibles
activity_rows = []
for name, entity in entities.items():
    df = entity.get("activities", pd.DataFrame())
    if isinstance(df, pd.DataFrame) and not df.empty:
        tmp = df.copy()
        tmp.insert(0, "Entreprise", name)
        activity_rows.append(tmp)

activites_csv_df = pd.concat(activity_rows, ignore_index=True) if activity_rows else pd.DataFrame()

# 2) Depuis KBO Public Search : on garde les tableaux contenant des codes NACE ou des mots d'activité.
activites_web = []
for name, data in kbo_pages.items():
    for idx, tab in enumerate(data.get("tables", [])):
        joined_cols = " ".join(map(str, tab.columns))
        joined_values = " ".join(tab.astype(str).head(20).to_numpy().ravel())
        if re.search(r"nace|activité|activiteit|onss|rsz", joined_cols + " " + joined_values, re.I):
            tmp = tab.copy()
            tmp.insert(0, "Entreprise", name)
            tmp.insert(1, "Source_table", idx)
            activites_web.append(tmp)

activites_web_df = pd.concat(activites_web, ignore_index=True) if activites_web else pd.DataFrame()

print("Activités depuis CSV KBO :")
display(activites_csv_df if not activites_csv_df.empty else pd.DataFrame({"message": ["Aucune activité trouvée dans les CSV locaux."]}))
print("Activités depuis KBO Public Search :")
display(activites_web_df if not activites_web_df.empty else pd.DataFrame({"message": ["Aucun tableau d'activités détecté sur les pages KBO."]}))


Activités depuis CSV KBO :


,Entreprise,EntityNumber,ActivityGroup,NaceVersion,NaceCode,Classification,ActivityGroup_libelle,NaceVersion_libelle,NaceCode_libelle
0,Google Belgium,0878.065.378,006,2025,73110,MAIN,Activités ONSS,2025,Activités des agences de publicité
1,Google Belgium,0878.065.378,006,2003,74401,MAIN,Activités ONSS,2003,Agences de publicité
2,Google Belgium,0878.065.378,001,2003,72600,MAIN,Activités TVA,2003,Autres activités rattachées à l'informatique
3,Google Belgium,0878.065.378,001,2025,62900,MAIN,Activités TVA,2025,Autres activités de service informatique
4,Google Belgium,0878.065.378,006,2008,73110,MAIN,Activités ONSS,2008,Activités des agences de publicité
5,Google Belgium,0878.065.378,001,2008,62090,MAIN,Activités TVA,2008,Autres activités informatiques
6,Apple Belgium,0836.157.420,006,2025,47400,MAIN,Activités ONSS,2025,Commerce de détail d’équipements de l’informat...
7,Apple Belgium,0836.157.420,001,2025,47400,MAIN,Activités TVA,2025,Commerce de détail d’équipements de l’informat...
8,Apple Belgium,0836.157.420,006,2008,47410,MAIN,Activités ONSS,2008,"Commerce de détail d'ordinateurs, d'unités pér..."
9,Apple Belgium,0836.157.420,001,2008,47410,MAIN,Activités TVA,2008,"Commerce de détail d'ordinateurs, d'unités pér..."


Activités depuis KBO Public Search :


,Entreprise,Source_table,0,1,2,3
0,Google Belgium,0,Algemeen,Algemeen,Algemeen,Algemeen
1,Google Belgium,0,Ondernemingsnummer:,0878.065.378,0878.065.378,0878.065.378
2,Google Belgium,0,Status:,Actief,Actief,Actief
3,Google Belgium,0,Rechtstoestand:,Normale toestand Sinds 21 december 2005,Normale toestand Sinds 21 december 2005,Normale toestand Sinds 21 december 2005
4,Google Belgium,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
118,SNCB,0,Btw 2025 49.110 - Personenvervoer per trein ...,Btw 2025 49.110 - Personenvervoer per trein ...,Btw 2025 49.110 - Personenvervoer per trein ...,NaN
119,SNCB,0,NaN,NaN,NaN,NaN
120,SNCB,0,Toon de activiteiten NACE-BEL-codes versie 2008.,Toon de activiteiten NACE-BEL-codes versie 2008.,Toon de activiteiten NACE-BEL-codes versie 2008.,NaN
121,SNCB,0,NaN,NaN,NaN,NaN


---
### 2.4) Dirigeants et Représentants

Liste de chaque dirigeant avec sa ou ses qualité(s).

| Nom | Qualité(s) |
|---|---|

In [11]:
# Extraction — Dirigeants et représentants
# Les dirigeants apparaissent généralement dans les tableaux KBO ; on détecte les tableaux contenant fonction/mandat/qualité.
dirigeants_rows = []
for name, data in kbo_pages.items():
    for idx, tab in enumerate(data.get("tables", [])):
        content = " ".join(map(str, tab.columns)) + " " + " ".join(tab.astype(str).head(20).to_numpy().ravel())
        if re.search(r"dirigeant|mandat|administrateur|gérant|qualité|functie|bestuurder|zaakvoerder", content, re.I):
            tmp = tab.copy()
            tmp.insert(0, "Entreprise", name)
            tmp.insert(1, "Source_table", idx)
            dirigeants_rows.append(tmp)

dirigeants_df = pd.concat(dirigeants_rows, ignore_index=True) if dirigeants_rows else pd.DataFrame(
    {"message": ["Aucun tableau de dirigeants détecté automatiquement. Vérifie les tableaux bruts dans kbo_pages[nom]['tables'] si besoin."]}
)
display(dirigeants_df)


,Entreprise,Source_table,0,1,2,3
0,Google Belgium,0,Algemeen,Algemeen,Algemeen,Algemeen
1,Google Belgium,0,Ondernemingsnummer:,0878.065.378,0878.065.378,0878.065.378
2,Google Belgium,0,Status:,Actief,Actief,Actief
3,Google Belgium,0,Rechtstoestand:,Normale toestand Sinds 21 december 2005,Normale toestand Sinds 21 december 2005,Normale toestand Sinds 21 december 2005
4,Google Belgium,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
118,SNCB,0,Btw 2025 49.110 - Personenvervoer per trein ...,Btw 2025 49.110 - Personenvervoer per trein ...,Btw 2025 49.110 - Personenvervoer per trein ...,NaN
119,SNCB,0,NaN,NaN,NaN,NaN
120,SNCB,0,Toon de activiteiten NACE-BEL-codes versie 2008.,Toon de activiteiten NACE-BEL-codes versie 2008.,Toon de activiteiten NACE-BEL-codes versie 2008.,NaN
121,SNCB,0,NaN,NaN,NaN,NaN


---
### 2.5) Liens entre Entités

Liste de chaque entité liée à l'entreprise.

| Entité | Numéro d'entreprise | Date du lien | Nature du lien | Statut actuel |
|---|---|---|---|---|

In [12]:
# Extraction — Liens entre entités
liens_rows = []
for name, data in kbo_pages.items():
    for idx, tab in enumerate(data.get("tables", [])):
        content = " ".join(map(str, tab.columns)) + " " + " ".join(tab.astype(str).head(20).to_numpy().ravel())
        if re.search(r"lien|entité liée|fusion|scission|relation|verbonden|link", content, re.I):
            tmp = tab.copy()
            tmp.insert(0, "Entreprise", name)
            tmp.insert(1, "Source_table", idx)
            liens_rows.append(tmp)

liens_df = pd.concat(liens_rows, ignore_index=True) if liens_rows else pd.DataFrame(
    {"message": ["Aucun lien entre entités détecté automatiquement sur les pages KBO."]}
)
display(liens_df)


,Entreprise,Source_table,0,1,2,3
0,Google Belgium,1,NaN,NaN,NaN,NaN
1,Google Belgium,1,Financiële gegevens,Financiële gegevens,Financiële gegevens,Financiële gegevens
2,Google Belgium,1,Kapitaal,"770.000,00 EUR","770.000,00 EUR","770.000,00 EUR"
3,Google Belgium,1,Jaarvergadering,juni,juni,juni
4,Google Belgium,1,Einddatum boekjaar,31 december,31 december,31 december
5,Google Belgium,1,NaN,NaN,NaN,NaN
6,Google Belgium,1,Linken tussen entiteiten,Linken tussen entiteiten,Linken tussen entiteiten,Linken tussen entiteiten
7,Google Belgium,1,Geen gegevens opgenomen in KBO.,Geen gegevens opgenomen in KBO.,Geen gegevens opgenomen in KBO.,NaN
8,Google Belgium,1,NaN,NaN,NaN,NaN
9,Google Belgium,1,Externe links,Externe links,Externe links,Externe links


---
### 2.6) Documents Juridiques (Statuts)

Source : https://statuts.notaire.be/stapor_v1/enterprise/{numero}/statutes

| Document | Date | Lien |
|---|---|---|

In [13]:
# Extraction — Documents juridiques (Statuts Notaire)
def scrape_statuts_notaire(num: str, max_pages: int = 20) -> pd.DataFrame:
    n = num_for_url(num)
    url = f"https://statuts.notaire.be/stapor_v1/enterprise/{n}/statutes"
    docs = []
    start = 0
    count = 20
    for _ in range(max_pages):
        try:
            r = requests.get(
                url,
                params={"enterpriseNumber": n, "statuteStart": start, "statuteCount": count},
                headers=HEADERS,
                timeout=30,
            )
            r.raise_for_status()
            data = r.json()
        except Exception as e:
            if not docs:
                return pd.DataFrame({"erreur": [str(e)], "url": [url]})
            break

        items = data if isinstance(data, list) else data.get("statutes") or data.get("items") or data.get("content") or []
        if not items:
            break
        for it in items:
            row = it if isinstance(it, dict) else {"document": str(it)}
            row["numero_entreprise"] = dotted_num(num)
            docs.append(row)
        if len(items) < count:
            break
        start += count
        time.sleep(0.25)

    return pd.DataFrame(docs) if docs else pd.DataFrame(columns=["numero_entreprise", "document", "date", "lien"])

statuts = {}
for name, num in ENTREPRISES.items():
    df = scrape_statuts_notaire(num)
    statuts[name] = df
    out = OUTPUT_DIR / f"statuts_{re.sub(r'\W+', '_', name.lower())}.csv"
    df.to_csv(out, index=False)
    print(f"{name}: {len(df)} ligne(s) -> {out}")

display(pd.concat([df.assign(Entreprise=name) for name, df in statuts.items()], ignore_index=True) if statuts else pd.DataFrame())


Google Belgium: 1 ligne(s) -> outputs\statuts_google_belgium.csv


Apple Belgium: 1 ligne(s) -> outputs\statuts_apple_belgium.csv


SNCB: 1 ligne(s) -> outputs\statuts_sncb.csv


,erreur,url,Entreprise
0,Expecting value: line 1 column 1 (char 0),https://statuts.notaire.be/stapor_v1/enterpris...,Google Belgium
1,Expecting value: line 1 column 1 (char 0),https://statuts.notaire.be/stapor_v1/enterpris...,Apple Belgium
2,Expecting value: line 1 column 1 (char 0),https://statuts.notaire.be/stapor_v1/enterpris...,SNCB


---
### 2.7) Comptes Annuels

Source : https://consult.cbso.nbb.be/consult-enterprise/{numero}

**Stratégie de stockage :**
- **PDFs** — tous les dépôts depuis l'an 2000 → stockés sur HDFS sous `/data/nbb/pdfs/{numero}/{annee}.pdf`
- **CSVs** — tous les dépôts depuis 2021 (format tabulaire) → stockés sur HDFS sous `/data/nbb/csvs/{numero}/{annee}.csv`

> Les dépôts antérieurs à 2021 n'ont pas de CSV disponible (fichiers migrés — PDF uniquement).
> Exclure les comptes **consolidés** (modèle `mc-*`) et dédupliquer à **un seul dépôt par année** (préférer FR sur NL).

| Année fiscale | Date de publication | PDF | CSV |
|---|---|---|---|

In [14]:
# Extraction — Comptes Annuels (NBB/CBSO) + stockage local/HDFS si disponible
# Remarque : le site CBSO/NBB est souvent une application dynamique. Le code ci-dessous collecte les liens visibles
# PDF/CSV si la page les expose dans le HTML. Sinon, utilise la page consult.cbso.nbb.be pour télécharger manuellement
# puis place les fichiers dans outputs/nbb/csvs/{numero}/{annee}.csv et outputs/nbb/pdfs/{numero}/{annee}.pdf.

CBSO_BASE = "https://consult.cbso.nbb.be/consult-enterprise"
NBB_DIR = OUTPUT_DIR / "nbb"
PDF_DIR = NBB_DIR / "pdfs"
CSV_DIR = NBB_DIR / "csvs"
PDF_DIR.mkdir(parents=True, exist_ok=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)


def cbso_consult_url(num: str) -> str:
    return f"{CBSO_BASE}/{num_for_url(num)}"


def collect_cbso_links(num: str) -> pd.DataFrame:
    url = cbso_consult_url(num)
    try:
        soup = get_soup(url)
    except Exception as e:
        return pd.DataFrame({"numero": [dotted_num(num)], "url": [url], "erreur": [str(e)]})

    rows = []
    for a in soup.find_all("a", href=True):
        href = requests.compat.urljoin(url, a["href"])
        text = clean_text(a.get_text(" "))
        if re.search(r"pdf|csv|xbrl|annual|accounts|comptes|jaarrekening", href + " " + text, re.I):
            year_match = re.search(r"(20\d{2}|19\d{2})", href + " " + text)
            rows.append({
                "numero": dotted_num(num),
                "annee": int(year_match.group(1)) if year_match else None,
                "texte": text,
                "lien": href,
                "type": "CSV" if ".csv" in href.lower() else "PDF" if ".pdf" in href.lower() else "AUTRE",
            })
    return pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame(columns=["numero", "annee", "texte", "lien", "type"])

cbso_links = {name: collect_cbso_links(num) for name, num in ENTREPRISES.items()}
cbso_links_df = pd.concat([df.assign(Entreprise=name) for name, df in cbso_links.items()], ignore_index=True)
display(cbso_links_df if not cbso_links_df.empty else pd.DataFrame({"message": ["Aucun lien CBSO détecté automatiquement. Télécharge les fichiers depuis la page CBSO indiquée dans la colonne URL."]}))

# URLs utiles à ouvrir manuellement si besoin
pd.DataFrame([{"Entreprise": name, "URL CBSO": cbso_consult_url(num)} for name, num in ENTREPRISES.items()])


,message
0,Aucun lien CBSO détecté automatiquement. Téléc...


,Entreprise,URL CBSO
0,Google Belgium,https://consult.cbso.nbb.be/consult-enterprise...
1,Apple Belgium,https://consult.cbso.nbb.be/consult-enterprise...
2,SNCB,https://consult.cbso.nbb.be/consult-enterprise...


---
### 2.8) Établissements

Liste de chaque établissement de l'entreprise.

| Numéro | Adresse | Date de création | Activité |
|---|---|---|---|

In [15]:
# Extraction — Établissements
establishment_rows = []
for name, entity in entities.items():
    df = entity.get("establishments", pd.DataFrame())
    if isinstance(df, pd.DataFrame) and not df.empty:
        tmp = df.copy()
        tmp.insert(0, "Entreprise", name)
        establishment_rows.append(tmp)

etablissements_df = pd.concat(establishment_rows, ignore_index=True) if establishment_rows else pd.DataFrame(
    {"message": ["Aucun établissement trouvé dans establishment.csv. Vérifie que les fichiers KBO sont bien chargés."]}
)
display(etablissements_df)


,Entreprise,EstablishmentNumber,StartDate,EnterpriseNumber
0,Google Belgium,2.151.627.472,10-02-2006,0878.065.378
1,Apple Belgium,2.200.008.005,06-05-2011,0836.157.420
2,Apple Belgium,2.244.110.935,19-09-2015,0836.157.420
3,SNCB,2.143.271.418,01-01-1968,0203.430.576
4,SNCB,2.143.272.111,01-01-1968,0203.430.576
...,...,...,...,...
268,SNCB,2.285.820.242,01-01-2019,0203.430.576
269,SNCB,2.291.131.882,01-05-2019,0203.430.576
270,SNCB,2.291.141.285,01-06-2019,0203.430.576
271,SNCB,2.297.380.365,01-12-2019,0203.430.576


---
### 2.9) Publications (eJustice)

Source : https://www.ejustice.just.fgov.be/cgi_tsv/list.pl?language=fr&btw={tva}&page=1

| Date | Référence (NUMAC) | Type | Lien |
|---|---|---|---|

In [16]:
# Extraction — Publications eJustice
def scrape_ejustice(num: str, max_pages: int = 50) -> pd.DataFrame:
    n = num_for_url(num)
    base = "https://www.ejustice.just.fgov.be/cgi_tsv/list.pl"
    pubs = []
    for page in range(1, max_pages + 1):
        try:
            soup = get_soup(base, params={"language": "fr", "btw": n, "page": page})
        except Exception as e:
            if not pubs:
                return pd.DataFrame({"numero": [dotted_num(num)], "erreur": [str(e)], "url": [f"{base}?language=fr&btw={n}&page=1"]})
            break

        found = 0
        for a in soup.find_all("a", href=True):
            href = requests.compat.urljoin(base, a["href"])
            around = clean_text(a.parent.get_text(" ") if a.parent else a.get_text(" "))
            # Recherche une publication avec date/référence NUMAC probable.
            if re.search(r"\d{2}/\d{2}/\d{4}|\d{4}-\d{2}-\d{2}|numac|moniteur|publication", around, re.I):
                pubs.append({"numero": dotted_num(num), "page": page, "texte": around, "lien": href})
                found += 1
        if found == 0:
            break
        time.sleep(0.25)

    return pd.DataFrame(pubs).drop_duplicates() if pubs else pd.DataFrame(columns=["numero", "page", "texte", "lien"])

publications = {}
for name, num in ENTREPRISES.items():
    df = scrape_ejustice(num)
    publications[name] = df
    out = OUTPUT_DIR / f"ejustice_{re.sub(r'\W+', '_', name.lower())}.csv"
    df.to_csv(out, index=False)
    print(f"{name}: {len(df)} publication(s) -> {out}")

publications_df = pd.concat([df.assign(Entreprise=name) for name, df in publications.items()], ignore_index=True)
display(publications_df)


Google Belgium: 195 publication(s) -> outputs\ejustice_google_belgium.csv


Apple Belgium: 120 publication(s) -> outputs\ejustice_apple_belgium.csv


SNCB: 938 publication(s) -> outputs\ejustice_sncb.csv


,numero,page,texte,lien,Entreprise
0,0878.065.378,1,Moniteur belge,https://www.ejustice.just.fgov.be/cgi/welcome....,Google Belgium
1,0878.065.378,1,"1) GOOGLE BELGIUM NV GOOGLEPLEX, STEENWEG OP E...",https://www.ejustice.just.fgov.be/cgi_tsv/arti...,Google Belgium
2,0878.065.378,1,"GOOGLEPLEX, STEENWEG OP ETTERBEEK 180 1040 BRU...",https://www.ejustice.just.fgov.be/tsv_pdf/2025...,Google Belgium
3,0878.065.378,1,"1) GOOGLE BELGIUM NV GOOGLEPLEX, STEENWEG OP E...",https://www.ejustice.just.fgov.be/cgi_tsv/arti...,Google Belgium
4,0878.065.378,1,"2) GOOGLE BELGIUM NV GOOGLEPLEX, STEENWEG OP E...",https://www.ejustice.just.fgov.be/cgi_tsv/arti...,Google Belgium
...,...,...,...,...,...
1248,0203.430.576,46,Moniteur belge,https://www.ejustice.just.fgov.be/cgi/welcome....,SNCB
1249,0203.430.576,47,Moniteur belge,https://www.ejustice.just.fgov.be/cgi/welcome....,SNCB
1250,0203.430.576,48,Moniteur belge,https://www.ejustice.just.fgov.be/cgi/welcome....,SNCB
1251,0203.430.576,49,Moniteur belge,https://www.ejustice.just.fgov.be/cgi/welcome....,SNCB


---
### 2.10) Informations de Contact

| Champ | Valeur |
|---|---|
| Téléphone | |
| Email | |
| Site web | |
| Adresse | |

In [17]:
# Extraction — Informations de Contact
contact_rows = []
for name, entity in entities.items():
    contacts = entity.get("contacts", pd.DataFrame())
    addresses = entity.get("addresses", pd.DataFrame())

    row = {"Entreprise": name, "Téléphone": "", "Email": "", "Site web": "", "Adresse": ""}

    if isinstance(contacts, pd.DataFrame) and not contacts.empty:
        flat = " | ".join(contacts.astype(str).to_numpy().ravel())
        phone = re.search(r"(\+?\d[\d\s./-]{7,}\d)", flat)
        email = re.search(r"[\w.\-+]+@[\w.\-]+\.\w+", flat)
        site = re.search(r"https?://\S+|www\.\S+", flat)
        row["Téléphone"] = phone.group(1) if phone else ""
        row["Email"] = email.group(0) if email else ""
        row["Site web"] = site.group(0) if site else ""

    if isinstance(addresses, pd.DataFrame) and not addresses.empty:
        row["Adresse"] = " | ".join(addresses.head(1).astype(str).to_numpy().ravel())
    else:
        # fallback depuis KBO Public Search
        web_row = infos_generales_df[infos_generales_df["Entreprise"] == name]
        if not web_row.empty:
            row["Adresse"] = web_row.iloc[0].get("Adresse", "")

    contact_rows.append(row)

contacts_df = pd.DataFrame(contact_rows)
display(contacts_df)


,Entreprise,Téléphone,Email,Site web,Adresse
0,Google Belgium,,,,0878.065.378 | REGO | | | 1040 | Brussel | B...
1,Apple Belgium,,,,0836.157.420 | REGO | | | 1210 | Sint-Joost-...
2,SNCB,,,,0203.430.576 | REGO | | | 1060 | Sint-Gillis...


---
## 3) Finances

Tableau financier depuis 2021 (ou depuis la création) jusqu'à 2025, enrichi avec l'EBIT.

### 📊 Tableau des indicateurs financiers

#### Performance
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Chiffre d'affaires | | | | | |
| Marge brute | | | | | |
| EBIT (Résultat d'exploitation) | | | | | |
| Résultat net | | | | | |

#### Croissance
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Taux de croissance du CA (%) | | | | | |
| Taux de marge brute (%) | | | | | |
| % de marge nette | | | | | |

#### Autonomie Financière
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Trésorerie | | | | | |
| Dettes financières | | | | | |
| Dette financière nette | | | | | |

#### Solvabilité
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Fonds propres | | | | | |

#### Ressources Humaines & Coûts
| Indicateur | 2021 | 2022 | 2023 | 2024 | 2025 |
|---|---|---|---|---|---|
| Nombre d'employés | | | | | |
| Coûts salariaux (salaires + charges + avantages) | | | | | |
| Employés en % du coût des revenus | | | | | |
| Revenu par employé | | | | | |
| Taxes payées | | | | | |

In [18]:
# ============================================================
# 3) Construction du tableau financier depuis les CSV NBB/CBSO
# ============================================================
ACCOUNT_CODES = {
    "Chiffre d'affaires": ["70"],
    "Achats / coût ventes": ["60"],
    "EBIT": ["9901"],
    "Résultat net": ["9904"],
    "Trésorerie": ["54/58", "54", "58"],
    "Dettes LT": ["17"],
    "Dettes CT": ["43"],
    "Fonds propres": ["10/15", "10", "11", "12", "13", "14", "15"],
    "Total actif": ["20/58"],
    "Employés": ["9087"],
    "Coûts salariaux": ["62"],
    "Taxes payées": ["67/77", "67"],
    "Amortissements": ["630"],
}


def find_amount_for_code(df: pd.DataFrame, codes: list[str]) -> Optional[float]:
    """Trouve le montant associé à un ou plusieurs codes comptables dans un CSV NBB."""
    if df.empty:
        return None

    code_col = find_column(df, ["Code", "Rubrique", "account", "Line", "Poste", "N°"])
    amount_col = find_column(df, ["Value", "Montant", "amount", "current", "Exercice", "Valeur", "2025", "2024", "2023", "2022", "2021"])

    if code_col is None:
        for c in df.columns:
            if df[c].astype(str).str.strip().isin(codes).any():
                code_col = c
                break

    if amount_col is None:
        numeric_scores = []
        for c in df.columns:
            s = pd.to_numeric(
                df[c].astype(str)
                     .str.replace(" ", "", regex=False)
                     .str.replace(".", "", regex=False)
                     .str.replace(",", ".", regex=False),
                errors="coerce",
            )
            numeric_scores.append((s.notna().sum(), c))
        numeric_scores = sorted(numeric_scores, reverse=True)
        amount_col = numeric_scores[0][1] if numeric_scores and numeric_scores[0][0] > 0 else None

    if code_col is None or amount_col is None:
        return None

    hit = df[df[code_col].astype(str).str.strip().isin(codes)].copy()
    if hit.empty:
        return None

    vals = pd.to_numeric(
        hit[amount_col].astype(str)
        .str.replace(" ", "", regex=False)
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )
    return float(vals.sum()) if vals.notna().any() else None


def build_financial_table(csv_by_year: dict[int, Path]) -> pd.DataFrame:
    rows = []
    previous_ca = None
    for year, path in sorted(csv_by_year.items()):
        if not Path(path).exists():
            print(f"⚠️ CSV manquant pour {year}: {path}")
            continue
        df = read_csv_flexible(Path(path))
        ca = find_amount_for_code(df, ACCOUNT_CODES["Chiffre d'affaires"])
        cout = find_amount_for_code(df, ACCOUNT_CODES["Achats / coût ventes"])
        ebit = find_amount_for_code(df, ACCOUNT_CODES["EBIT"])
        rn = find_amount_for_code(df, ACCOUNT_CODES["Résultat net"])
        treso = find_amount_for_code(df, ACCOUNT_CODES["Trésorerie"])
        dettes_lt = find_amount_for_code(df, ACCOUNT_CODES["Dettes LT"]) or 0
        dettes_ct = find_amount_for_code(df, ACCOUNT_CODES["Dettes CT"]) or 0
        fonds = find_amount_for_code(df, ACCOUNT_CODES["Fonds propres"])
        total_actif = find_amount_for_code(df, ACCOUNT_CODES["Total actif"])
        emp = find_amount_for_code(df, ACCOUNT_CODES["Employés"])
        salaires = find_amount_for_code(df, ACCOUNT_CODES["Coûts salariaux"])
        taxes = find_amount_for_code(df, ACCOUNT_CODES["Taxes payées"])

        marge_brute = None if ca is None or cout is None else ca - cout
        dettes_fin = dettes_lt + dettes_ct
        dette_nette = None if treso is None else dettes_fin - treso
        croissance = None if previous_ca in [None, 0] or ca is None else (ca - previous_ca) / previous_ca * 100
        if ca is not None:
            previous_ca = ca

        rows.append({
            "Année": year,
            "Chiffre d'affaires": ca,
            "Marge brute": marge_brute,
            "EBIT": ebit,
            "Résultat net": rn,
            "Taux de croissance du CA (%)": croissance,
            "Taux de marge brute (%)": None if ca in [None, 0] or marge_brute is None else marge_brute / ca * 100,
            "% de marge nette": None if ca in [None, 0] or rn is None else rn / ca * 100,
            "Trésorerie": treso,
            "Dettes financières": dettes_fin,
            "Dette financière nette": dette_nette,
            "Fonds propres": fonds,
            "Autonomie financière (%)": None if total_actif in [None, 0] or fonds is None else fonds / total_actif * 100,
            "Nombre d'employés": emp,
            "Coûts salariaux": salaires,
            "Employés en % du coût des revenus": None if cout in [None, 0] or salaires is None else salaires / cout * 100,
            "Revenu par employé": None if emp in [None, 0] or ca is None else ca / emp,
            "Taxes payées": taxes,
        })
    return pd.DataFrame(rows)


def csv_paths_for_company(company_key: str, numero: str) -> dict[int, Path]:
    """Retourne les chemins attendus des CSV NBB pour 2021-2025."""
    folder_candidates = [
        CSV_DIR / num_for_url(numero),
        CSV_DIR / company_key,
        OUTPUT_DIR / "nbb_csvs" / company_key,
        Path("/data/nbb/csvs") / num_for_url(numero),
    ]
    result = {}
    for year in range(2021, 2026):
        for folder in folder_candidates:
            p = folder / f"{year}.csv"
            if p.exists():
                result[year] = p
                break
        else:
            # chemin par défaut à créer/télécharger
            result[year] = folder_candidates[0] / f"{year}.csv"
    return result

financial_tables = {}
for name, num in ENTREPRISES.items():
    key = re.sub(r"\W+", "_", name.lower()).strip("_")
    paths = csv_paths_for_company(key, num)
    financial_tables[name] = build_financial_table(paths)
    print(f"\n{name} — chemins CSV attendus :")
    for y, p in paths.items():
        print(f"  {y}: {p}")

# Affichage du premier tableau disponible, sinon une table de rappel.
available = {k: v for k, v in financial_tables.items() if not v.empty}
if available:
    for name, df in available.items():
        display(Markdown(f"### Tableau financier — {name}"))
        display(df)
else:
    display(pd.DataFrame({
        "Action à faire": [
            "Télécharger les CSV CBSO/NBB 2021-2025 pour chaque entreprise",
            "Les placer dans outputs/nbb/csvs/{numero_sans_zero_initial}/{annee}.csv",
            "Relancer cette cellule pour calculer les indicateurs financiers",
        ]
    }))


⚠️ CSV manquant pour 2021: outputs\nbb\csvs\878065378\2021.csv
⚠️ CSV manquant pour 2022: outputs\nbb\csvs\878065378\2022.csv
⚠️ CSV manquant pour 2023: outputs\nbb\csvs\878065378\2023.csv
⚠️ CSV manquant pour 2024: outputs\nbb\csvs\878065378\2024.csv
⚠️ CSV manquant pour 2025: outputs\nbb\csvs\878065378\2025.csv

Google Belgium — chemins CSV attendus :
  2021: outputs\nbb\csvs\878065378\2021.csv
  2022: outputs\nbb\csvs\878065378\2022.csv
  2023: outputs\nbb\csvs\878065378\2023.csv
  2024: outputs\nbb\csvs\878065378\2024.csv
  2025: outputs\nbb\csvs\878065378\2025.csv
⚠️ CSV manquant pour 2021: outputs\nbb\csvs\836157420\2021.csv
⚠️ CSV manquant pour 2022: outputs\nbb\csvs\836157420\2022.csv
⚠️ CSV manquant pour 2023: outputs\nbb\csvs\836157420\2023.csv
⚠️ CSV manquant pour 2024: outputs\nbb\csvs\836157420\2024.csv
⚠️ CSV manquant pour 2025: outputs\nbb\csvs\836157420\2025.csv

Apple Belgium — chemins CSV attendus :
  2021: outputs\nbb\csvs\836157420\2021.csv
  2022: outputs\nbb\csvs\

,Action à faire
0,Télécharger les CSV CBSO/NBB 2021-2025 pour ch...
1,Les placer dans outputs/nbb/csvs/{numero_sans_...
2,Relancer cette cellule pour calculer les indic...


In [19]:
# Graphe Sankey — visualisation du compte de résultats pour chaque entreprise

def plot_sankey(fin: pd.DataFrame, name: str, year: int = 2025):
    if go is None:
        print("Installe plotly pour afficher ce graphe : pip install plotly")
        return None
    if fin.empty or year not in fin["Année"].values:
        print(f"Pas de données financières pour {name} en {year}.")
        return None

    r = fin[fin["Année"] == year].iloc[0]
    ca = r.get("Chiffre d'affaires") or 0
    marge = r.get("Marge brute") or 0
    ebit = r.get("EBIT") or 0
    rn = r.get("Résultat net") or 0

    cout_ventes = max(ca - marge, 0) if ca and marge else 0
    charges_exploitation = max(marge - ebit, 0) if marge and ebit else 0
    impots_financier = max(ebit - rn, 0) if ebit and rn else 0

    labels = ["CA", "Coût des ventes", "Marge brute", "Charges d'exploitation", "EBIT", "Impôts / financier", "Résultat net"]
    source = [0, 0, 2, 2, 4, 4]
    target = [1, 2, 3, 4, 5, 6]
    values = [cout_ventes, marge, charges_exploitation, ebit, impots_financier, rn]

    fig = go.Figure(data=[go.Sankey(node={"label": labels}, link={"source": source, "target": target, "value": values})])
    fig.update_layout(title_text=f"{name} — Sankey du compte de résultats {year}", font_size=12)
    out = OUTPUT_DIR / f"sankey_{re.sub(r'\W+', '_', name.lower())}_{year}.html"
    fig.write_html(out)
    print(f"Graphe exporté : {out}")
    fig.show()
    return fig

for name, fin in financial_tables.items():
    plot_sankey(fin, name, 2025)


Installe plotly pour afficher ce graphe : pip install plotly
Installe plotly pour afficher ce graphe : pip install plotly
Installe plotly pour afficher ce graphe : pip install plotly


---
### 📈 Graphiques — Chiffre d'affaires & Résultat net (2021–2025)

In [20]:
# Graphiques — Chiffre d'affaires & Résultat net (2021–2025)
def plot_ca_resultat(fin: pd.DataFrame, name: str):
    if px is None:
        print("Installe plotly pour afficher ce graphe : pip install plotly")
        return None
    if fin.empty:
        print(f"Pas de données financières pour {name}.")
        return None
    cols = [c for c in ["Chiffre d'affaires", "Résultat net"] if c in fin.columns]
    long = fin.melt(id_vars="Année", value_vars=cols, var_name="Indicateur", value_name="Valeur")
    fig = px.line(long, x="Année", y="Valeur", color="Indicateur", markers=True, title=f"{name} — CA et Résultat net")
    out = OUTPUT_DIR / f"ca_resultat_{re.sub(r'\W+', '_', name.lower())}.html"
    fig.write_html(out)
    print(f"Graphe exporté : {out}")
    fig.show()
    return fig

for name, fin in financial_tables.items():
    plot_ca_resultat(fin, name)


Installe plotly pour afficher ce graphe : pip install plotly
Installe plotly pour afficher ce graphe : pip install plotly
Installe plotly pour afficher ce graphe : pip install plotly


In [21]:
# Export final des tableaux produits par le TP
exports = []

def export_df(df: pd.DataFrame, filename: str):
    if isinstance(df, pd.DataFrame) and not df.empty:
        path = OUTPUT_DIR / filename
        df.to_csv(path, index=False)
        exports.append(str(path))

export_df(infos_generales_df, "infos_generales.csv")
export_df(infos_juridiques_df, "infos_juridiques.csv")
export_df(activites_csv_df, "activites_kbo_csv.csv")
export_df(activites_web_df, "activites_kbo_web.csv")
export_df(dirigeants_df, "dirigeants.csv")
export_df(liens_df, "liens_entites.csv")
export_df(etablissements_df, "etablissements.csv")
export_df(publications_df, "publications_ejustice.csv")
export_df(contacts_df, "contacts.csv")
export_df(cbso_links_df, "liens_cbso.csv")

for name, fin in financial_tables.items():
    export_df(fin, f"finance_{re.sub(r'\W+', '_', name.lower())}.csv")

print("Fichiers exportés :")
for path in exports:
    print("-", path)


Fichiers exportés :
- outputs\infos_generales.csv
- outputs\infos_juridiques.csv
- outputs\activites_kbo_csv.csv
- outputs\activites_kbo_web.csv
- outputs\dirigeants.csv
- outputs\liens_entites.csv
- outputs\etablissements.csv
- outputs\publications_ejustice.csv
- outputs\contacts.csv


---
## 📖 Glossaire des indicateurs financiers

Tous les indicateurs ci-dessous sont calculés à partir des **codes comptables NBB** (Plan Comptable Minimum Normalisé).
Les codes entre parenthèses correspondent aux lignes du CSV téléchargé depuis la NBB.

---

### Performance

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Chiffre d'affaires (CA)** | Revenu total généré par la vente de biens et services sur l'exercice | — | `70` |
| **Marge brute** | Ce qu'il reste après déduction du coût direct des marchandises vendues | CA − Coût des marchandises | `70 − 60` |
| **EBIT** *(Résultat d'exploitation)* | Bénéfice avant intérêts et impôts — mesure la performance opérationnelle pure | — | `9901` |
| **EBITDA** | EBIT avant amortissements — proxy du cash généré par l'exploitation | EBIT + Amortissements | `9901 + 630` |
| **Résultat net** | Bénéfice final après toutes les charges, intérêts et impôts | — | `9904` |

---

### Croissance & Marges

| Indicateur | Définition | Formule |
|---|---|---|
| **Taux de croissance du CA (%)** | Variation du chiffre d'affaires d'une année sur l'autre | `(CA_n − CA_{n-1}) / CA_{n-1} × 100` |
| **Taux de marge brute (%)** | Part de la marge brute dans le CA — reflète la rentabilité commerciale | `Marge brute / CA × 100` |
| **Marge nette (%)** | Part du résultat net dans le CA — résume la rentabilité globale | `Résultat net / CA × 100` |
| **Marge EBITDA (%)** | Capacité à générer du cash avant investissements et fiscalité | `EBITDA / CA × 100` |

---

### Autonomie Financière

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Trésorerie** | Liquidités disponibles immédiatement (banque + caisse) | — | `54/58` |
| **Dettes financières** | Total des emprunts bancaires (long terme + court terme) | Dettes LT + Dettes CT | `17 + 43` |
| **Dette financière nette** | Endettement réel après déduction de la trésorerie disponible. Négatif = position de trésorerie nette (bonne santé) | Dettes financières − Trésorerie | `(17 + 43) − 54/58` |

---

### Solvabilité

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Fonds propres** | Capital apporté par les actionnaires + bénéfices accumulés non distribués | — | `10/15` |
| **Total actif** | Ensemble des ressources économiques contrôlées par l'entreprise | — | `20/58` |
| **Autonomie financière (%)** | Part des actifs financée par les fonds propres (sans recours à la dette). Plus c'est élevé, plus l'entreprise est indépendante | `Fonds propres / Total actif × 100` |

---

### Ressources Humaines

| Indicateur | Définition | Formule | Code(s) |
|---|---|---|---|
| **Effectif FTE moyen** | Nombre moyen d'équivalents temps plein sur l'exercice | — | `9087` |
| **Coûts salariaux** | Rémunérations brutes + cotisations patronales + avantages extra-légaux | — | `62` |
| **Revenu par ETP** | CA généré en moyenne par chaque employé — mesure la productivité | `CA / Effectif FTE` |
| **Coût moyen par ETP** | Coût salarial total rapporté à un équivalent temps plein | `Coûts salariaux / Effectif FTE` |
| **Part salariale dans le CA (%)** | Poids des charges de personnel dans le chiffre d'affaires | `Coûts salariaux / CA × 100` |
| **Taxes payées** | Impôt des sociétés (ISOC) effectivement payé sur l'exercice | — | `670/3` |